# 03. Classify your own image with DoReFaNet

Put a JPG, PNG or BMP image in the **images** folder beside this notebook. Run cells in order with **Shift + Enter**. The newest image is selected by default, or you can enter its filename in section 3.

This model predicts one of 1,000 ImageNet classes for the **whole image**; it does not locate multiple objects. [Notebook 02](02_dorefanet-imagenet-samples.ipynb) uses labelled samples instead.

![DoReFaNet topology](dorefanet-topology.svg)

The first convolution and fully connected layers run on the ARM CPU, and the intermediate quantized layers on the FPGA. Shut down other FPGA notebook kernels first. Keep swap enabled if needed on the 512 MB board. Use the installed package with the unused `Xlnk` imports already removed; no reinstall is required.

## 1. Locate installed parameters and your images

Run this cell. It checks files before initializing the FPGA. `NOTEBOOK_DIR` normally follows the notebook working directory, with the standard board folder as a fallback. Upload your images into its **images** subfolder, then rerun this cell.

In [ ]:
import os
import gc
import pickle
import random
import time
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.ticker import PercentFormatter
from PIL import Image
from IPython.display import display

import qnn
from qnn import Dorefanet, utils

QNN_ROOT = Path(qnn.__file__).resolve().parent
QNN_PARAM_DIR = QNN_ROOT / "params"
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "imagenet-classes.pkl").is_file():
    NOTEBOOK_DIR = Path("/home/xilinx/jupyter_notebooks/qnn")
SAMPLE_DIR = NOTEBOOK_DIR / "images"
LABEL_FILE = NOTEBOOK_DIR / "imagenet-classes.pkl"

required_files = [QNN_PARAM_DIR / name for name in (
    "dorefanet-layers.json", "dorefanet-conv0.npy",
    "dorefanet-fc-normalized.npy", "ilsvrc_pp_mean.npy")]
required_files.append(LABEL_FILE)
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Required files missing:\n" + "\n".join(missing))
if not SAMPLE_DIR.is_dir():
    raise FileNotFoundError("Create/upload images into: " + str(SAMPLE_DIR))
image_files = sorted(path for path in SAMPLE_DIR.iterdir()
                     if path.is_file() and path.suffix.lower() in
                     (".jpg", ".jpeg", ".png", ".bmp"))
if not image_files:
    raise FileNotFoundError("Upload a JPG/PNG/BMP image into " + str(SAMPLE_DIR))
print("Python package:", QNN_ROOT)
print("Notebook data:", NOTEBOOK_DIR)
print("Available images:", len(image_files))

## 2. Load weights and class labels, then initialize the model

Run the next cell and wait for **Model ready**. It loads the existing NumPy weight dictionaries and 1,000 ImageNet class labels, then initializes the FPGA accelerator. It uses the package's default network configuration, which already follows the installed package location.

A message is printed before each large load. If this cell fails, do not continue to the inference cells. Rerunning it releases a previously initialized classifier before creating another one. The pickle and NumPy object files here are the trusted files supplied by this package.

In [ ]:
if "classifier" in globals():
    classifier.deinit_accelerator()
    del classifier
# Avoid holding old weight dictionaries while reloading this cell.
for variable in ("conv0_weights", "fc_weights"):
    globals().pop(variable, None)
gc.collect()

print("Loading first-convolution weights...", flush=True)
conv0_weights = np.load(str(QNN_PARAM_DIR / "dorefanet-conv0.npy"),
                       encoding="latin1", allow_pickle=True).item()
print("Loading fully connected weights; please wait...", flush=True)
fc_weights = np.load(str(QNN_PARAM_DIR / "dorefanet-fc-normalized.npy"),
                    encoding="latin1", allow_pickle=True).item()
for key in ("conv0/W", "conv0/T"):
    if key not in conv0_weights:
        raise KeyError("Missing weight: " + key)
for key in ("fc0/Wn", "fc0/bn", "fc1/Wn", "fc1/bn", "fct/W"):
    if key not in fc_weights:
        raise KeyError("Missing weight: " + key)

with LABEL_FILE.open("rb") as label_stream:
    classes = pickle.load(label_stream, encoding="latin1")
if set(classes.keys()) != set(range(1000)):
    raise ValueError("Expected ImageNet labels indexed from 0 to 999.")
names = {index: classes[index][1].split(",")[0] for index in classes}
synset_to_index = {classes[index][0]: index for index in classes}

print("Initializing the FPGA...", flush=True)
classifier = Dorefanet()
try:
    net = classifier.load_network()
    classifier.init_accelerator()
except BaseException:
    classifier.deinit_accelerator()
    raise
print("Model ready. ImageNet classes:", len(classes))

## 3. Select and display your image

Leave `IMAGE_NAME = None` to use the newest image in **images**. To select another file, replace `None` with its quoted filename. Run this cell, then sections **4–8**. These custom images have no trusted ground-truth label, so the results are predictions rather than measured accuracy.

In [ ]:
IMAGE_NAME = None  # Example: replace None with an existing filename string.
image_path = max(image_files, key=lambda path: path.stat().st_mtime) if IMAGE_NAME is None else SAMPLE_DIR / IMAGE_NAME
if not image_path.is_file():
    raise FileNotFoundError("Image not found: " + str(image_path))
img, img_class = classifier.load_image(str(image_path))
expected_index = None  # No ground-truth label for a user image.
print("Image:", image_path.name)
print("Correct class:", names[expected_index] if expected_index is not None
      else "not available from this filename")
with Image.open(str(image_path)) as opened:
    preview = opened.convert("RGB")
preview.thumbnail((480, 360))
display(preview)
stage_seconds = {}

## 4. First convolution on the ARM CPU

Run the next cell. This layer uses the original full-precision weights. Its output is thresholded to the representation required by the quantized FPGA layers. The displayed time includes this convolution and thresholding.

In [ ]:
start = time.perf_counter()
conv0 = utils.conv_layer(img, conv0_weights["conv0/W"], stride=4)
conv0 = utils.threshold(conv0, conv0_weights["conv0/T"])
stage_seconds["First convolution + threshold (CPU)"] = time.perf_counter() - start
print("CPU first stage: {:.3f} s".format(
    stage_seconds["First convolution + threshold (CPU)"]))

## 5. Quantized layers on the FPGA

Run the next cell. The same accelerator executes the configured sequence of quantized layers. Buffer preparation and output conversion are outside the timed inference call.

This timing describes the **offload call**, including its runtime overhead, rather than isolated FPGA arithmetic cycles.

In [ ]:
out_dim = net["merge4"]["output_dim"]
out_ch = net["merge4"]["output_channels"]
conv_output = classifier.get_accel_buffer(out_ch, out_dim)
conv_input = np.ascontiguousarray(classifier.prepare_buffer(conv0))
start = time.perf_counter()
classifier.inference(conv_input, conv_output)
stage_seconds["Quantized-layer offload (FPGA)"] = time.perf_counter() - start
conv_output_hw_raw = conv_output.copy()
conv_output = classifier.postprocess_buffer(conv_output)
if not np.all(np.isfinite(conv_output)):
    raise ValueError("Non-finite accelerator output. Check model initialization.")
print("FPGA offload call: {:.3f} s".format(
    stage_seconds["Quantized-layer offload (FPGA)"]))

## 6. Fully connected layers on the ARM CPU

Run the next cell. It keeps the original network's normalization and fully connected computations. A zero accelerator output is reported explicitly, so invalid results are not passed through a division by zero.

In [ ]:
scale = np.max(conv_output)
if scale <= 0:
    raise RuntimeError("Accelerator output is all zero. Stop and check initialization before continuing.")
fc_input = conv_output / scale
if fc_input.size != fc_weights["fc0/Wn"].shape[0]:
    raise ValueError("Accelerator output size does not match the fully connected weights.")
start = time.perf_counter()
fc0_out = utils.fully_connected(fc_input, fc_weights["fc0/Wn"], fc_weights["fc0/bn"])
fc0_out = utils.quantize(utils.qrelu(fc0_out), 2)
fc1_out = utils.fully_connected(fc0_out, fc_weights["fc1/Wn"], fc_weights["fc1/bn"])
fc1_out = utils.qrelu(fc1_out)
fct_b = np.zeros((fc_weights["fct/W"].shape[1],))
fct_out = utils.fully_connected(fc1_out, fc_weights["fct/W"], fct_b)
stage_seconds["Fully connected layers (CPU)"] = time.perf_counter() - start
print("CPU fully connected stages: {:.3f} s".format(
    stage_seconds["Fully connected layers (CPU)"]))

## 7. Read the five predictions

The bars show the five highest ImageNet softmax scores for this image, from the combined CPU/FPGA pipeline. They do not measure accuracy and are not the output of a separate software-only model. An object absent from the training classes will still receive a class prediction.

In [ ]:
if np.asarray(fct_out).shape != (len(classes),) or not np.all(np.isfinite(fct_out)):
    raise ValueError("Expected 1,000 finite classification scores.")
out = utils.softmax(fct_out)
topn = utils.get_topn_indexes(out, 5)
print("Correct class:", names[expected_index] if expected_index is not None else "unknown")
for index in topn:
    print("{:25s} {:7.2%}".format(names[int(index)], float(out[index])))
if expected_index is not None:
    print("Top-1:", "correct" if int(topn[0]) == expected_index else "wrong")
    print("Top-5:", "correct class included" if expected_index in topn
          else "correct class not included")
fig, ax = plt.subplots(figsize=(8, 3.5))
positions = np.arange(len(topn))
colors = ["green" if int(index) == expected_index else "steelblue" for index in topn]
ax.barh(positions, out[topn], color=colors)
ax.set_yticks(positions)
ax.set_yticklabels([names[int(index)] for index in topn])
ax.invert_yaxis()
ax.set_xlim(0, min(1.0, float(np.max(out[topn])) * 1.25))
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Softmax score (axis scaled to the displayed predictions)")
ax.set_title("Top-5 predictions for your image")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()
plt.close(fig)

## 8. Compare measured stage times

Run the next cell. It shows the three measured computation stages and prints their sum. The two CPU stages and the FPGA stage execute **different layers**: this is a division of work, not a CPU-versus-FPGA speedup measurement for the same calculation.

The sum excludes image loading, buffer preparation/conversion, softmax, plotting, and time spent waiting between notebook cells. It must not be presented as camera FPS or complete application throughput. No timing file is shared with other notebooks.

In [ ]:
stage_order = ["First convolution + threshold (CPU)",
               "Quantized-layer offload (FPGA)",
               "Fully connected layers (CPU)"]
if any(name not in stage_seconds for name in stage_order):
    raise RuntimeError("Run all three computation stages for the current image first.")
values = [stage_seconds[name] for name in stage_order]
for name, seconds in zip(stage_order, values):
    print("{}: {:.3f} s".format(name, seconds))
print("Sum of measured stages: {:.3f} s".format(sum(values)))
fig, ax = plt.subplots(figsize=(9, 3.5))
positions = np.arange(len(stage_order))
ax.barh(positions, values, color=["steelblue", "darkorange", "steelblue"])
ax.set_yticks(positions)
ax.set_yticklabels(stage_order)
ax.invert_yaxis()
ax.set_xlabel("Measured stage time (seconds)")
ax.set_title("CPU and FPGA stages of the same inference")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

## 9. Optional: compare the same quantized layers in software

Leave `RUN_SOFTWARE_COMPARISON = False` to skip this slower test. Set it to `True` to execute the **same quantized-layer input** using the installed C++ software backend, without downloading another bitstream. It compares the raw outputs and reports the two inference-call times. The first convolution and fully connected layers are excluded from both times; this is not a second end-to-end classification.

Run this section before cleanup. The software backend is always released, even if its call fails.

In [ ]:
RUN_SOFTWARE_COMPARISON = False
if RUN_SOFTWARE_COMPARISON:
    classifier_sw = Dorefanet("python_sw")
    try:
        classifier_sw.load_network()
        classifier_sw.init_accelerator(bit_file=None)
        sw_output = classifier_sw.get_accel_buffer(out_ch, out_dim)
        start = time.perf_counter()
        classifier_sw.inference(conv_input, sw_output)
        sw_seconds = time.perf_counter() - start
        hw_seconds = stage_seconds["Quantized-layer offload (FPGA)"]
        print("Same quantized layers — FPGA call: {:.3f} s; software call: {:.3f} s".format(
            hw_seconds, sw_seconds))
        print("Raw outputs exactly equal:", np.array_equal(conv_output_hw_raw, sw_output))
        if hw_seconds > 0:
            print("Software / FPGA call time: {:.2f}x".format(sw_seconds / hw_seconds))
    finally:
        classifier_sw.deinit_accelerator()
        del classifier_sw
        globals().pop("sw_output", None)
else:
    print("Software comparison skipped.")

## 10. Finish and release resources

Run the next cell when you have finished testing. It calls the model's own cleanup method and drops large intermediate arrays and weight dictionaries. No `Xlnk` import or global device reset is used.

To run again after cleanup, start from section 2. Shut down this kernel before another FPGA notebook.

In [ ]:
if "classifier" in globals():
    classifier.deinit_accelerator()
    del classifier
for variable in ("conv0_weights", "fc_weights", "img", "conv0", "conv_input",
                 "conv_output", "conv_output_hw_raw", "fc_input", "fc0_out", "fc1_out", "fct_out"):
    globals().pop(variable, None)
gc.collect()
print("Classifier released and large Python arrays cleared.")

**Validation:** Python flow and error handling are checked off-board with simulated accelerator/native interfaces and the original NumPy network helpers. Actual FPGA execution, native Darknet behavior and prediction quality must still be checked on the PYNQ board.